In [ ]:
# @title RSI 4 Trendline Scanner (Clean Output & Auto-Fix)
# @markdown **INSTRUCTIONS:** Run this cell. If it crashes/restarts, **wait 5 seconds and RUN IT AGAIN.**
# @markdown The first run fixes the Google Colab environment. The second run performs the scan.

import sys
import subprocess
import os
import importlib

# --- 1. ENVIRONMENT AUTO-FIX (NumPy < 2.0) ---
def fix_environment():
    needs_restart = False
    
    # Check for pandas_ta and yfinance
    for lib in ['pandas_ta', 'yfinance']:
        try:
            importlib.import_module(lib)
        except ImportError:
            print(f"Installing {lib}...")
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', lib, '-q'])
            needs_restart = True

    # Check for NumPy Compatibility (Must be < 2.0 for Colab's SciPy)
    try:
        import numpy
        if int(numpy.__version__.split('.')[0]) >= 2:
            print(f"Downgrading NumPy from {numpy.__version__} to 1.26.4...")
            subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy', '-q'])
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numpy==1.26.4', '-q'])
            needs_restart = True
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numpy==1.26.4', '-q'])
        needs_restart = True

    if needs_restart:
        print("Environment patched. RESTARTING KERNEL... (Please run this cell again!)")
        time.sleep(1)
        os.kill(os.getpid(), 9)

# --- 2. WARNING SUPPRESSION (Clean Output) ---
import warnings
import logging
import time

# Run the fix
fix_environment()

# Suppress Warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)

# Suppress YFinance Logging
logger = logging.getLogger('yfinance')
logger.setLevel(logging.CRITICAL)
logger.propagate = False

# --- 3. IMPORTS ---
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from scipy.signal import argrelextrema
from tqdm import tqdm

# --- 4. CONFIGURATION ---
RSI_PERIOD = 4        
RSI_OB_LEVEL = 80     
RSI_OS_LEVEL = 20     
LOOKBACK_BARS = 150   

# Entry Zones
SELL_ZONE_LOW = 60
SELL_ZONE_HIGH = 70
BUY_ZONE_LOW = 30
BUY_ZONE_HIGH = 40

# Tickers
tickers_string = "A AA AAL AAP AAPL ABBV ABNB ABT ACN ADBE ADI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BITO BK BKR BMY BP BSX BX BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIA DIS DKNG DLR DOCU DOW DRI DVN DXC EA EBAY ED EEM EFA EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EWJ EWW EWY EWZ EXC EXPE F FANG FAST FCX FDX FE FI FITB FOXA FSLR FTI FTV FXE FXI GD GDX GE GILD GLD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HBI HCA HD HIG HLT HOG HOLX HON HPE HPQ HYG IBB IBIT IBM ICE IEF INTC IP IPG IRM IVZ IWM IYR JCI JD JETS JNJ JNPR JPM K KHC KMI KO KR KRE KSS KWEB LEN LI LKQ LLY LNC LOW LQD LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX NVDA NWL NWSA O OIH OKE OMC ON ORCL OXY PARA PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON PYPL QCOM QQQ RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHD SCHW SEDG SHOP SIRI SLB SLV SMCI SMH SNAP SNOW SO SOFI SOXL SOXS SPG SPX SPXL SPXS SPY SQQQ STX SWKS SYF SYY T TAP TBT TCOM TDOC TFC TGT TJX TLT TMO TMUS TPR TQQQ TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNG UNH UNP UPS UPST URBN USB USO UVXY V VALE VFC VGK VTR VXX VZ WAB WBA WBD WDC WFC WM WMB WMT WU WY WYNN X XBI XEL XHB XLB XLC XLE XLF XLI XLK XLP XLRE XLU XLV XLY XOM XOP XRT XRX XSP XYZ YELP YINN ZION ZM"
tickers = tickers_string.split()

buys = []
sells = []

print(f"Scanning {len(tickers)} stocks.")
print(f"Generating 4-Hour Bars from Hourly Data...")

# Scan loop
for ticker in tqdm(tickers):
    try:
        # 1. Download Data (Suppressed Output via auto_adjust=True explicit)
        # We catch exceptions to prevent logs from spilling over
        try:
            df_1h = yf.download(ticker, period="6mo", interval="1h", progress=False, threads=False, auto_adjust=True)
        except Exception:
            continue

        if len(df_1h) < 200: continue 
        if isinstance(df_1h.columns, pd.MultiIndex): df_1h.columns = df_1h.columns.get_level_values(0)

        # 2. RESAMPLE TO 4-HOUR CHART
        logic = {'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'}
        df = df_1h.resample('4h').apply(logic).dropna()
        if len(df) < LOOKBACK_BARS: continue

        # 3. RSI Calc
        df['RSI'] = ta.rsi(df['Close'], length=RSI_PERIOD)
        
        current_rsi = df['RSI'].iloc[-1]
        prev_rsi = df['RSI'].iloc[-2]
        rsi_window = df['RSI'].values[-LOOKBACK_BARS:]
        
        # ==========================================
        # SELL LOGIC (RSI 60-70) -> Bear Call Spread
        # ==========================================
        if SELL_ZONE_LOW <= current_rsi <= SELL_ZONE_HIGH:
            
            # 1. Slope: Must be pointing DOWN
            if current_rsi >= prev_rsi: continue 

            # 2. Find Recent Overbought Peak (>80)
            ob_indices = np.where(rsi_window > RSI_OB_LEVEL)[0]
            if len(ob_indices) > 0:
                last_ob_idx = ob_indices[-1] 
                
                # 3. 50-LINE BARRIER CHECK
                # From the Peak to NOW, RSI must NOT have crossed below 50.
                post_peak_slice = rsi_window[last_ob_idx:]
                if np.min(post_peak_slice) < 50:
                    continue 

                if last_ob_idx < len(rsi_window) - 1:
                    # 4. Find Start (Oversold < 20) BEFORE peak
                    pre_peak_slice = rsi_window[:last_ob_idx]
                    os_indices = np.where(pre_peak_slice < RSI_OS_LEVEL)[0]
                    
                    if len(os_indices) > 0:
                        last_os_idx = os_indices[-1] 
                        
                        # 5. Trendline Logic 
                        trend_section = rsi_window[last_os_idx:last_ob_idx]
                        local_min_indices = argrelextrema(trend_section, np.less)[0]
                        
                        valid_anchors = []
                        for idx in local_min_indices:
                            val = trend_section[idx]
                            # Touch must be in Neutral Zone
                            if val > RSI_OS_LEVEL and val < RSI_OB_LEVEL:
                                valid_anchors.append(idx)
                                
                        if len(valid_anchors) > 0:
                            anchor_rel_idx = valid_anchors[-1]
                            
                            x1 = last_os_idx
                            y1 = rsi_window[x1]
                            x2 = last_os_idx + anchor_rel_idx
                            y2 = rsi_window[x2]
                            
                            if x2 > x1:
                                m = (y2 - y1) / (x2 - x1)
                                c = y1 - (m * x1)
                                
                                # Check Violation
                                curr_idx = len(rsi_window) - 1
                                projected_rsi = (m * curr_idx) + c
                                
                                if current_rsi < projected_rsi:
                                    sells.append({
                                        'Ticker': ticker,
                                        'RSI': round(current_rsi, 2),
                                        'Slope': "DOWN",
                                        'Gap': round(projected_rsi - current_rsi, 2),
                                        'Touches': len(valid_anchors)
                                    })

        # ==========================================
        # BUY LOGIC (RSI 30-40) -> Bull Put Spread
        # ==========================================
        elif BUY_ZONE_LOW <= current_rsi <= BUY_ZONE_HIGH:
            
            # 1. Slope: Must be pointing UP
            if current_rsi <= prev_rsi: continue

            # 2. Find Recent Oversold Trough (<20)
            os_indices = np.where(rsi_window < RSI_OS_LEVEL)[0]
            if len(os_indices) > 0:
                last_os_idx = os_indices[-1] 
                
                # 3. 50-LINE BARRIER CHECK
                # From the Trough to NOW, RSI must NOT have crossed above 50.
                post_trough_slice = rsi_window[last_os_idx:]
                if np.max(post_trough_slice) > 50:
                    continue 

                if last_os_idx < len(rsi_window) - 1:
                    # 4. Find Start (Overbought > 80) BEFORE trough
                    pre_trough_slice = rsi_window[:last_os_idx]
                    ob_indices = np.where(pre_trough_slice > RSI_OB_LEVEL)[0]
                    
                    if len(ob_indices) > 0:
                        last_ob_idx = ob_indices[-1]
                        
                        # 5. Trendline Logic
                        trend_section = rsi_window[last_ob_idx:last_os_idx]
                        local_max_indices = argrelextrema(trend_section, np.greater)[0]
                        
                        valid_anchors = []
                        for idx in local_max_indices:
                            val = trend_section[idx]
                            # Touch must be in Neutral Zone
                            if val > RSI_OS_LEVEL and val < RSI_OB_LEVEL:
                                valid_anchors.append(idx)
                        
                        if len(valid_anchors) > 0:
                            anchor_rel_idx = valid_anchors[-1]
                            
                            x1 = last_ob_idx
                            y1 = rsi_window[x1]
                            x2 = last_ob_idx + anchor_rel_idx
                            y2 = rsi_window[x2]
                            
                            if x2 > x1:
                                m = (y2 - y1) / (x2 - x1)
                                c = y1 - (m * x1)
                                
                                # Check Violation
                                curr_idx = len(rsi_window) - 1
                                projected_rsi = (m * curr_idx) + c
                                
                                if current_rsi > projected_rsi:
                                    buys.append({
                                        'Ticker': ticker,
                                        'RSI': round(current_rsi, 2),
                                        'Slope': "UP",
                                        'Gap': round(current_rsi - projected_rsi, 2),
                                        'Touches': len(valid_anchors)
                                    })

    except Exception as e:
        continue

# --- SORTING & OUTPUT ---
buys.sort(key=lambda x: (x['Touches'], x['Gap']), reverse=True)
sells.sort(key=lambda x: (x['Touches'], x['Gap']), reverse=True)

top_10_buys = buys[:10]
top_10_sells = sells[:10]

print("\n" + "="*70)
print(f"TOP 10 BULLISH SETUPS (For PUT Spreads)")
print(f"Criteria: 4H Chart | RSI 4 | Break of Resistance | RSI < 50")
print("="*70)
print(f"{'TICKER':<10} {'RSI':<10} {'SLOPE':<10} {'TOUCHES':<10} {'GAP SCORE'}")
print("-" * 70)
for t in top_10_buys:
    print(f"{t['Ticker']:<10} {t['RSI']:<10} {t['Slope']:<10} {t['Touches']:<10} {t['Gap']}")

print("\n" + "="*70)
print(f"TOP 10 BEARISH SETUPS (For CALL Spreads)")
print(f"Criteria: 4H Chart | RSI 4 | Break of Support | RSI > 50")
print("="*70)
print(f"{'TICKER':<10} {'RSI':<10} {'SLOPE':<10} {'TOUCHES':<10} {'GAP SCORE'}")
print("-" * 70)
for t in top_10_sells:
    print(f"{t['Ticker']:<10} {t['RSI']:<10} {t['Slope']:<10} {t['Touches']:<10} {t['Gap']}")

# TradingView Strings
buy_str = ",".join([t['Ticker'] for t in top_10_buys])
sell_str = ",".join([t['Ticker'] for t in top_10_sells])

print("\n" + "="*70)
print("TRADINGVIEW IMPORT LISTS")
print("="*70)
print("TOP 10 BULLISH:(For PUT Spreads)")
print(buy_str)
print("-" * 70)
print("TOP 10 BEARISH:(For CALL Spreads)")
print(sell_str)